<a href="https://colab.research.google.com/github/rishithareddym/AI-Lab/blob/main/1BF24CS181_WEEK7_UNIFICATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1BF24C181 -- Muddam Rishitha Reddy

import re

def is_variable(x):
    """Defines variables as tokens matching x, y, z, X, Y, or Z."""
    return isinstance(x, str) and x.strip() in ['x', 'y', 'z', 'X', 'Y', 'Z']

def parse_expr(expr_str):
    """Converts a string representation into a nested list structure."""
    expr_str = expr_str.replace(" ", "")
    match = re.match(r"^(\w+)\((.*)\)$", expr_str)
    if not match:
        return expr_str  # Atomic literal (constant or variable)

    predicate = match.group(1)
    args_str = match.group(2)

    args = []
    current = []
    depth = 0
    for char in args_str:
        if char == '(':
            depth += 1
        elif char == ')':
            depth -= 1
        if char == ',' and depth == 0:
            args.append(parse_expr("".join(current)))
            current = []
        else:
            current.append(char)
    if current:
        args.append(parse_expr("".join(current)))

    return [predicate] + args

def stringify(expr):
    """Converts nested list structure back to a readable string format."""
    if isinstance(expr, list):
        return f"{expr[0]}({', '.join(stringify(arg) for arg in expr[1:])})"
    return str(expr)

def occur_check(var, term):
    """Returns True if the variable exists inside the nested functional term."""
    if var == term:
        return True
    if isinstance(term, list):
        return any(occur_check(var, t) for t in term[1:])
    return False

def apply_sub(sub, expr):
    """Recursively replaces variables using the current substitutions map."""
    if is_variable(expr):
        return sub.get(expr, expr)
    if isinstance(expr, list):
        return [expr[0]] + [apply_sub(sub, arg) for arg in expr[1:]]
    return expr

def unify(E1, E2, theta=None):
    """Executes the First-Order Logic Unification algorithm."""
    if theta is None:
        theta = {}

    E1 = apply_sub(theta, E1)
    E2 = apply_sub(theta, E2)

    if E1 == E2:
        return theta

    if is_variable(E1):
        if occur_check(E1, E2):
            return "FAIL (Occur Check)"
        new_theta = {E1: E2}
        for k, v in theta.items():
            theta[k] = apply_sub(new_theta, v)
        theta[E1] = E2
        return theta

    if is_variable(E2):
        if occur_check(E2, E1):
            return "FAIL (Occur Check)"
        new_theta = {E2: E1}
        for k, v in theta.items():
            theta[k] = apply_sub(new_theta, v)
        theta[E2] = E1
        return theta

    if isinstance(E1, list) and isinstance(E2, list):
        if E1[0] != E2[0] or len(E1) != len(E2):
            return f"FAIL (Mismatch between '{E1[0]}' and '{E2[0]}')"

        for i in range(1, len(E1)):
            theta = unify(E1[i], E2[i], theta)
            if isinstance(theta, str) and theta.startswith("FAIL"):
                return theta
        return theta

    return f"FAIL (Cannot match {stringify(E1)} with {stringify(E2)})"

# --- Interactive User Input Loop ---
if __name__ == "__main__":
    print("=== Interactive First-Order Logic Unifier ===")
    print("Variables allowed: x, y, z, X, Y, Z")
    print("Example formats: P(f(a),g(y)) or Q(a,x)")
    print("Type 'exit' to quit.\n")

    while True:
        exp1_input = input("Enter Expression 1: ").strip()
        if exp1_input.lower() == 'exit':
            break

        exp2_input = input("Enter Expression 2: ").strip()
        if exp2_input.lower() == 'exit':
            break

        try:
            p1 = parse_expr(exp1_input)
            p2 = parse_expr(exp2_input)

            res = unify(p1, p2)

            print("\n" + "-"*40)
            if isinstance(res, dict):
                print("✅ Result: Unification Successful!")
                print("   Most General Unifier (MGU):")
                if not res:
                    print("     (None - Expressions are identical)")
                for var, val in res.items():
                    print(f"     {var} -> {stringify(val)}")
            else:
                print(f"❌ Result: Unification Failed")
                print(f"   Reason: {res}")
            print("-"*40 + "\n")

        except Exception as e:
            print(f"\n⚠️ Error parsing expression. Check your brackets. ({e})\n")


=== Interactive First-Order Logic Unifier ===
Variables allowed: x, y, z, X, Y, Z
Example formats: P(f(a),g(y)) or Q(a,x)
Type 'exit' to quit.

Enter Expression 1:  P(f(a), g(y))
Enter Expression 2:  P(X, X)

----------------------------------------
❌ Result: Unification Failed
   Reason: FAIL (Mismatch between 'g' and 'f')
----------------------------------------

